In [1]:
import pandas as pd

raw = pd.Series(["2026-10-24","bad",None,"2026-02-30"], dtype="object")
parsed = pd.to_datetime(raw, format="%Y-%m-%d", errors="coerce")
print(parsed)
# 第一项为有效日期，后三项为 NaT

failed = raw.notna() & parsed.isna()
print(failed.tolist()) # [False,True,False,True]
print(parsed.dt.year)
print(parsed.dt.dayofweek) # 周一=0，周日=6；2026-10-24 是5

0   2026-10-24
1          NaT
2          NaT
3          NaT
dtype: datetime64[us]
[False, True, False, True]
0    2026.0
1       NaN
2       NaN
3       NaN
dtype: float64
0    5.0
1    NaN
2    NaN
3    NaN
dtype: float64


In [3]:
start = pd.Timestamp("2026-10-23 23:00:00")
end = pd.Timestamp("2026-10-24 01:00:00")
elapsed = end-start
print(elapsed.total_seconds()/3600) # 2.0 小时
print(elapsed.days) # 0，不是跨了一个日历日期就算完整一天
elapsed

2.0
0


Timedelta('0 days 02:00:00')

In [4]:
df = pd.DataFrame({
    "student":["A","B","C","D"],
    "score":[90,90,95,90],
    "minutes":[30,20,25,20],
}, index=["r8","r2","r9","r1"])
work = df.copy()
work["_pos"] = range(len(work))
ranked = work.sort_values(
    ["score","minutes","_pos"],
    ascending=[False,True,True],
).drop(columns="_pos")
print(ranked["student"].tolist()) # ["C","B","D","A"]

['C', 'B', 'D', 'A']


In [5]:
times = pd.Series(pd.to_datetime(
    ["2026-10-24 00:00:00","2026-10-24 23:59:59","2026-10-25 00:00:00"],
    format="%Y-%m-%d %H:%M:%S",
))
mask = (times>=pd.Timestamp("2026-10-24")) & (times<pd.Timestamp("2026-10-25"))
print(mask.tolist()) # [True,True,False]

[True, True, False]


In [6]:
events = pd.DataFrame({
    "order_id":["A","B","A","B","A"],
    "updated_at":["2026-10-23 09:00:00","2026-10-24 08:00:00",
                  "2026-10-24 10:00:00","bad","2026-10-24 10:00:00"],
    "amount":[10,20,15,99,18],
}, index=["e1","e2","e3","e4","e5"])
work = events.copy()
work["updated_at"] = pd.to_datetime(
    work["updated_at"], format="%Y-%m-%d %H:%M:%S", errors="coerce"
)
work["_pos"] = range(len(work))
valid = work.loc[work["updated_at"].notna()]
latest = (
    valid.sort_values(["updated_at","_pos"], ascending=[True,True])
    .drop_duplicates(subset=["order_id"], keep="last")
    .sort_values("order_id")
    .drop(columns="_pos")
)
print(latest[["order_id","amount"]])
# A:18（同时间选原输入更靠后的 e5），B:20（非法时间 e4 排除）

   order_id  amount
e5        A      18
e2        B      20


1. 日期解析与质量标记
输入列顺序 ["event_id","date_text"]，date_text 为 object dtype，元素是 str/None。 返回原列后追加 date、status、month、is_weekend：

非缺失文本先严格检查形如4位年-2位月-2位日（ASCII数字），不strip，不接受额外空格；再按 %Y-%m-%d 用 pd.to_datetime(errors="coerce") 检查日期有效性。
原始None：status="missing"；非缺失但格式或日期非法："invalid"；成功："valid"。status为object。
date 为无时区datetime列，无效/缺失为NaT；不强制具体时间精度。
month 为Pandas Int64，可缺失；is_weekend为Pandas boolean，周六/日True，工作日False，无效/缺失pd.NA。
不删除任何记录，保留原序和索引。空表输出同样列结构。
样例 date_text： ["2026-10-24","2026-10-26","2024-02-29","2026-02-29",None,"2026-1-2"," 2026-10-24"] status→["valid","valid","valid","invalid","missing","invalid","invalid"]。 前三条month为10/10/2；前三条is_weekend为True/False/False，其余缺失。 硬验收：样例、空串、真实文本"None"、合法闰日/非法闰日、原始缺失、空表及单行；严格列序与dtype；date用isna核对；输入不变。可用Series.str.fullmatch做格式预检，正则只是辅助不展开专题。

In [26]:
def check(s:pd.Series):
    if pd.isna(s):
        return 'missing'
    elif pd.isna(pd.to_datetime(s,format='%Y-%m-%d',errors='coerce')):
        return 'invalid'
    else:
        return 'valid'
def month(s:pd.Series):
    if pd.isna(s):
        return pd.NA
    else:
        return pd.to_datetime(s).month
def is_weekend(s:pd.Series):
    if pd.notna(s):
        return pd.to_datetime(s).dayofweek>=5
    else:
        return pd.NA
def parse_event_dates(df: pd.DataFrame) -> pd.DataFrame:
    work=df.copy()
    work['date_text']=pd.to_datetime(work['date_text'],format='%Y-%m-%d',errors='coerce')
    work_2=work.assign(
        date=work['date_text'].dt.date,
        status=df['date_text'].apply(check),
    )
    work_3=work_2.assign(
        month=work_2['date'].apply(month),
        is_weekend=work_2['date'].apply(is_weekend),
    )
    return work_3
parse_event_dates(pd.DataFrame({
    'event_id':['e1','e2','e3','e4','e5','e6','e7'],
    'date_text':['2026-10-24','2026-10-26','2024-02-29','2026-02-29',None,'2026-1-2',' 2026-10-24']
}))

,event_id,date_text,date,status,month,is_weekend
0,e1,2026-10-24,2026-10-24,valid,10,True
1,e2,2026-10-26,2026-10-26,valid,10,False
2,e3,2024-02-29,2024-02-29,valid,2,False
3,e4,NaT,NaT,invalid,<NA>,<NA>
4,e5,NaT,NaT,missing,<NA>,<NA>
5,e6,2026-01-02,2026-01-02,valid,1,False
6,e7,NaT,NaT,invalid,<NA>,<NA>


2. 排行榜与并列
输入列顺序 ["student","score","minutes"]；student为非缺失str，score/minutes为float64。 score可缺失，非缺失须在[0,100]；minutes全部非缺失且>=0。非法数值抛ValueError；所有数值有限且不含bool。 排除score缺失行，按score降序、minutes升序，完全并列按原输入顺序；保留原列和dtype，原索引不变、不添加名次列。 样例 student=["A","B","C","D"]，score=[90,90,95,90]，minutes=[30,20,25,20]→student=["C","B","D","A"]。 硬验收：完全并列不按姓名重排；无序/非默认索引；空表/全缺失score；0/100边界；-1/101成绩、负/缺失minutes抛ValueError（即使该行score也缺失）。输入不变。

In [30]:
def rank_attempts(df: pd.DataFrame) -> pd.DataFrame:
    work=df.copy()
    work['poc']=range(len(work))
    if ((work['score']<0) | (work['score']>100) | (work['minutes']<0) | (work['minutes'].isna())).any():
        raise ValueError
    work=work[work['score'].notna()]
    ranked=work.sort_values(
        ['score','minutes','poc'],
        ascending=[False,True,True],
    ).drop(columns='poc')
    return ranked['student'].tolist()
rank_attempts(pd.DataFrame({
    "student":["A","B","C","D"],
    "score":[90,90,95,90],
    "minutes":[30,20,25,20]
}))


['C', 'B', 'D', 'A']

3. 每订单保留最新版本
输入列顺序 ["order_id","updated_at","amount"]。order_id为非空、非缺失str；amount为有限float64，无需范围校验；updated_at为object，元素str/None。

时间文本先严格匹配 YYYY-MM-DD HH:MM:SS（ASCII数字），再按 %Y-%m-%d %H:%M:%S 解析errors="coerce"，无时区；不strip，不接受日期缩写或额外空白。
无效/缺失时间行不参与版本选择；某订单全无效则不输出该订单。
每个order_id选择时间最大行；同时间多条选原输入位置最后一条，不依据索引标签大小或amount。
最终按order_id原字符串字典序升序，保留被选行的原索引。
输出且仅输出原三列，updated_at替换为解析后的datetime列，其他dtype保持。
必须使用sort_values和drop_duplicates完成核心版本选择；不修改输入；空表/全无效返回结构正确空表。
样例见notes.md例题3：最终索引e5/e2，A金额18，B金额20。 硬验收：乱序输入、同时间并列、无效时间位于末尾、全无效订单、空表、一个订单、不同索引标签顺序；同一业务键多条而非全行相同时仍正确判重；不存在辅助列。使用assert_frame_equal比较，日期精度可归一后核对。

In [44]:
def latest_orders(df: pd.DataFrame) -> pd.DataFrame:
    work=df.copy()
    ok=(pd.isna(work['updated_at']) | work['updated_at']\
        .str.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}"))
    work['datetime']=pd.to_datetime(
        work[ok]['updated_at'],
        format='%Y-%m-%d %H:%M:%S',
        errors='coerce'
    )
    work['pos']=range(len(work))
    work=work.dropna(subset='datetime')
    ranked=work.sort_values(
        ['order_id','pos'],
        ascending=[True, True]
    ).drop_duplicates(subset='order_id',keep='last').drop(columns='pos')
    return ranked[['order_id','datetime','amount']]
events = pd.DataFrame({
    "order_id":["A","B","A","B","A"],
    "updated_at":["2026-10-23 09:00:00","2026-10-24 08:00:00",
                  "2026-10-24 10:00:00","bad","2026-10-24 10:00:00"],
    "amount":[10,20,15,99,18],
}, index=["e1","e2","e3","e4","e5"])
latest_orders(events)

,order_id,datetime,amount
e5,A,2026-10-24 10:00:00,18
e2,B,2026-10-24 08:00:00,20
